# Resolve a CSV batch to product URLs

This notebook runs every input row through the same in-process resolver and canonical acceptance policy.

There is no UI, API server, Docker stack, browser service, queue, polling layer, `nest_asyncio`, or runtime patching.

Input contract:

| Column | Required |
|---|---:|
| `main_text` | Yes |
| `country_code` | Yes |
| `row_id` | No |
| `retailer_name` | No |
| `ean` | No |
| `language_code` | No |

## One-time setup

```bash
conda env create -f environment.yml
conda activate product-url-notebook
python -m playwright install chromium
cp .env.example .env
jupyter lab
```

Set `SERPAPI_API_KEY` in `.env`.

In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the web_search_tool repository.")

ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=False)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

if not os.getenv("SERPAPI_API_KEY", "").strip():
    raise EnvironmentError("SERPAPI_API_KEY is missing from .env")

print(f"Repository root: {ROOT}")
print("Browser mode: local Playwright")

## Batch configuration

The output CSV is checkpointed after every row. Completed rows remain available if execution is interrupted.

In [ ]:
INPUT_CSV = ROOT / "samples" / "products.csv"
OUTPUT_CSV = ROOT / "data" / "results" / "product_urls.csv"
FEATURE_SET = "toy"

RUNTIME_OPTIONS = {
    "search_credits": 3,
    "results_per_search": 20,
    "max_candidates": 12,
    "max_per_domain": 2,
    "max_workers": 6,
    "browser_enabled": True,
    "browser_required": True,
    "browser_candidates": 6,
    "reasoning_enabled": False,
    "reasoning_required": False,
}

CONTINUE_ON_TECHNICAL_FAILURE = True

In [ ]:
input_df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)

aliases = {
    "MAIN_TEXT": "main_text",
    "COUNTRY": "country_code",
    "RETAILER": "retailer_name",
    "EAN": "ean",
}
input_df = input_df.rename(columns={
    name: aliases[name] for name in aliases if name in input_df.columns
})

missing = {"main_text", "country_code"} - set(input_df.columns)
if missing:
    raise ValueError(f"Input CSV is missing mandatory columns: {sorted(missing)}")

for optional in ("row_id", "retailer_name", "ean", "language_code"):
    if optional not in input_df.columns:
        input_df[optional] = ""

input_df[[
    "row_id", "main_text", "country_code",
    "retailer_name", "ean", "language_code"
]]

In [ ]:
from product_url_v2 import (
    ProductInput,
    ProductURLOrchestrator,
    evaluate_acceptance,
    load_config,
)

config = load_config(ROOT / "config" / "default.json")
resolver = ProductURLOrchestrator(config)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
rows = []

for index, source in input_df.iterrows():
    row_id = source["row_id"].strip() or f"ROW-{index + 1:06d}"
    product = ProductInput(
        row_id=row_id,
        main_text=source["main_text"],
        country_code=source["country_code"],
        retailer_name=source["retailer_name"] or None,
        ean=source["ean"] or None,
        language_code=source["language_code"] or None,
        feature_set=FEATURE_SET,
        runtime_options=RUNTIME_OPTIONS,
    )

    print(f"[{index + 1}/{len(input_df)}] {row_id}: {product.main_text[:80]}")
    result = resolver.resolve(product)
    selected = next(
        (
            candidate
            for candidate in result.candidates
            if candidate.candidate_id == result.decision.selected_candidate_id
        ),
        None,
    )
    verdict = evaluate_acceptance(selected) if selected else None

    rows.append({
        "ROW_ID": row_id,
        "MAIN_TEXT": product.main_text,
        "COUNTRY": product.country_code,
        "RETAILER": product.retailer_name or "",
        "EAN": product.ean or "",
        "PRODUCT_URL": result.decision.selected_url or "",
        "DELIVERY_STATUS": result.decision.status.value,
        "CONFIDENCE": round(result.decision.confidence, 4),
        "CODING_READY": result.decision.coding_ready,
        "SOURCE_ROLE": selected.source_role.value if selected else "",
        "IDENTITY_STATUS": selected.identity_match.value if selected else "",
        "IDENTIFIER_VERIFIED": verdict.identifier_verified if verdict else False,
        "DIRECT_PRODUCT_PAGE": selected.direct_product_page.value if selected else "",
        "BROWSER_ACCESS": selected.browser_access.value if selected else "",
        "TEXT_EXTRACTABLE": selected.text_extractable.value if selected else "",
        "JUSTIFICATION": " ".join(result.decision.reasons),
        "WARNINGS": " | ".join(result.decision.warnings),
        "ARTIFACT_DIR": result.artifact_dir,
        "ELAPSED_MS": result.elapsed_ms,
        "TECHNICAL_ERROR": result.technical_error,
    })

    pd.DataFrame(rows).to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    if (
        result.decision.status.value == "TECHNICAL_FAILURE"
        and not CONTINUE_ON_TECHNICAL_FAILURE
    ):
        raise RuntimeError(result.technical_error or f"{row_id} failed technically")

print(f"Saved: {OUTPUT_CSV}")

## Batch output

A delivered row must have status `VERIFIED` or `REVIEW_REQUIRED` and exactly one `PRODUCT_URL`.

In [ ]:
output_df = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)
output_df

In [ ]:
output_df.groupby(
    "DELIVERY_STATUS",
    dropna=False,
).size().rename("ROWS").reset_index().sort_values("ROWS", ascending=False)

## Batch acceptance assertion

In [ ]:
successful = output_df["DELIVERY_STATUS"].isin(["VERIFIED", "REVIEW_REQUIRED"])
has_url = output_df["PRODUCT_URL"].str.strip().ne("")

invalid = output_df[successful.ne(has_url)]
assert invalid.empty, (
    "Successful rows must contain one URL, "
    "and failed rows must not contain a delivered URL."
)

print(f"Validated {len(output_df)} output rows.")
print(f"Output CSV: {OUTPUT_CSV}")